# Pass 3: Actions Demanded
LLM coding of actions called for in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Emotions (Passes 1–2) explain *how* the public feels; Pass 3 captures
*what they want done* — the actionable demand layer (investigate, resign,
compensate, reform, punish, "None"/implicit).

### Approach
Same chunked protocol with `../prompts/pass-03-actions-demanded.md`.
Single string `action_demanded` per comment; "None" or implicit-demand
description when no explicit call.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1–4** | Same as Pass 1 | Load → slim/clean → chunk (15) → LLM call + `json.loads` |
| **5. Export** | Save + inspect | `sample_pass03_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *Which emotions (Passes 1–2) predict which demands?*
- *Do demands escalate across phases (onset → lingering)?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass03_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 30
chunk_size = 15
sample_seed = 42
model = "gemma-2-9b-it"
overwrite = False

In [ ]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
OUTPUT_PATH = f"../output/{crisis_name}/sample_pass03_results.jsonl"
LOG_PATH    = f"../output/{crisis_name}/sample_pass03_failures.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-2-9b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 512
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)